# Zero-to-Hero: Production-Grade AI SQL Assistant with LangChain + Ollama

This notebook builds and explains a full local AI SQL analytics assistant.

## Learning Objectives
- Understand architecture for enterprise-style NL-to-SQL assistants.
- Build schema-aware SQL generation with LangChain and direct prompting.
- Add read-only safety validation and guardrails.
- Execute SQL, explain it, and visualize results.
- Benchmark and evaluate model quality with LLM-as-a-judge.


## Why These Choices?

### Why SQLite for learning?
- Fully local, reproducible, zero cloud dependency.
- Simple setup with strong SQL feature coverage for analytics.
- Easy packaging for tutorials and CI.

### Why LangChain `SQLDatabase`?
- Standard schema-aware abstraction.
- Faster bootstrapping for text-to-SQL pipelines.
- Good baseline for comparison against direct prompting.

### Why Ollama?
- Local inference for privacy and cost control.
- Deterministic settings possible (`temperature=0`, fixed seed).
- Easy model swapping (`qwen3.5:4b`, `granite4.1:3b`).

### Why deterministic decoding (`temperature=0`)?
- SQL must be reproducible for audits and regression tests.
- Lower syntax drift and fewer random unsafe tokens.
- Stable benchmark comparisons and easier debugging.

### Why read-only access?
- Protects production-like data from destructive writes.
- Enforces analytics-only assistant behavior.
- Required for enterprise governance.

### Why schema-aware prompting?
- Reduces hallucinated tables/columns.
- Improves join accuracy and KPI grounding.
- Enables business-term to schema mapping.


## System Architecture

```mermaid
flowchart LR
    User[User Question] --> UI[Streamlit App]
    UI --> Pipeline[AISQLAssistant]
    Pipeline --> Schema[Schema Cache + Glossary]
    Pipeline --> GenLC[LangChain SQL Generation]
    Pipeline --> GenDirect[Direct Ollama SQL Generation]
    GenLC --> Validate[SQL Validation Guardrails]
    GenDirect --> Validate
    Validate --> Execute[Read-only SQLite Execution]
    Execute --> Explain[SQL Explanation + Business Interpretation]
    Execute --> Viz[Visualization Recommender]
    Pipeline --> Eval[Benchmark + Judge]
    Pipeline --> Memory[Conversation + History Store]
```


In [1]:
# 1) Imports and runtime setup
from ai_sql_assistant.logging_utils import configure_logging
from ai_sql_assistant.config import get_settings
from ai_sql_assistant.data.northwind import build_northwind_databases
from ai_sql_assistant.schema.introspector import inspect_database, save_schema_report, generate_erd
from ai_sql_assistant.pipeline.assistant import AISQLAssistant
from ai_sql_assistant.types import QueryRequest

configure_logging()
settings = get_settings()
settings


AppSettings(ollama_host=HttpUrl('http://127.0.0.1:11434/'), models=ModelConfig(generator_model='qwen3.5:4b', comparison_model='granite4.1:3b', judge_model='granite4.1:3b', temperature=0.0, top_p=1.0, num_predict=256, timeout_seconds=300, request_retries=2, retry_backoff_seconds=2.0), database=DatabaseConfig(raw_db_path=PosixPath('/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/LLMs, RAG, and Smart Search/Create an AI SQL Assistant with LangChain/data/sqlite/northwind_raw.db'), scaled_db_path=PosixPath('/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/LLMs, RAG, and Smart Search/Create an AI SQL Assistant with LangChain/data/sqlite/northwind_scaled.db'), app_state_db_path=PosixPath('/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/LLMs, RAG, and Smart Search/Create an AI SQL Assistant with LangChain/data/sqlite/app_state.db'), active_db='scaled'), safety=SafetyConfig(max_rows=5000, max_query_seconds=30.0, allow_multi_statement=False, strict_join_checks=True), evaluation=

In [2]:
# 2) Build realistic relational dataset (Northwind raw + scaled)
# This step attempts public Northwind download first; falls back to deterministic synthetic build.
build_result = build_northwind_databases(
    raw_db_path=settings.database.raw_db_path,
    scaled_db_path=settings.database.scaled_db_path,
    scale_factor=8,
)
build_result


2026-06-28 20:08:15 | INFO     | ai_sql_assistant.data.northwind:_try_download_sqlite:322 | Downloaded Northwind from https://raw.githubusercontent.com/jpwhite3/northwind-SQLite3/main/dist/northwind.db


BuildResult(raw_db_path=PosixPath('/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/LLMs, RAG, and Smart Search/Create an AI SQL Assistant with LangChain/data/sqlite/northwind_raw.db'), scaled_db_path=PosixPath('/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/LLMs, RAG, and Smart Search/Create an AI SQL Assistant with LangChain/data/sqlite/northwind_scaled.db'), source='downloaded_incompatible_schema_fallback_synthetic', raw_orders=12000, scaled_orders=96000)

In [3]:
# 3) Automatic database exploration and schema report generation
report = inspect_database(settings.database.active_db_path)
md_path, json_path = save_schema_report(report)
erd_path = generate_erd(report)

print(f"Schema markdown: {md_path}")
print(f"Schema json: {json_path}")
print(f"ERD image: {erd_path}")
print("Tables:", list(report['tables'].keys()))


2026-06-28 20:08:20 | INFO     | ai_sql_assistant.schema.introspector:save_schema_report:196 | Saved schema markdown report to /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/LLMs, RAG, and Smart Search/Create an AI SQL Assistant with LangChain/artifacts/reports/schema_report.md


2026-06-28 20:08:20 | INFO     | ai_sql_assistant.schema.introspector:save_schema_report:197 | Saved schema JSON report to /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/LLMs, RAG, and Smart Search/Create an AI SQL Assistant with LangChain/artifacts/reports/schema_report.json


2026-06-28 20:08:20 | INFO     | ai_sql_assistant.schema.introspector:generate_erd:227 | Saved ERD diagram to /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/LLMs, RAG, and Smart Search/Create an AI SQL Assistant with LangChain/artifacts/diagrams/schema_erd.png


Schema markdown: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/LLMs, RAG, and Smart Search/Create an AI SQL Assistant with LangChain/artifacts/reports/schema_report.md
Schema json: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/LLMs, RAG, and Smart Search/Create an AI SQL Assistant with LangChain/artifacts/reports/schema_report.json
ERD image: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/LLMs, RAG, and Smart Search/Create an AI SQL Assistant with LangChain/artifacts/diagrams/schema_erd.png
Tables: ['categories', 'customers', 'employees', 'order_details', 'orders', 'products', 'shippers', 'suppliers']


### What we explored automatically
- Tables, columns, datatypes.
- Primary keys and foreign keys.
- Relationship graph and indexes.
- Row counts and null statistics.
- Sample rows for quick inspection.


In [4]:
# 4) Initialize assistant pipeline
assistant = AISQLAssistant(settings)


In [5]:
# 5) Ask a business question end-to-end
request = QueryRequest(
    question="Show monthly net revenue for Europe in 2024.",
    persona="Finance",
    user_id="notebook",
    conversation_id="nb-session",
)
response = assistant.ask(request, approach="langchain", model=settings.models.generator_model)

print("Status:", response.execution.status)
print("SQL:\n", response.execution.sql)
print("Rows:", response.execution.row_count)
print("Execution ms:", response.execution.execution_time_ms)


Status: success
SQL:
 SELECT STRFTIME('%Y-%m', o.order_date) AS month, SUM(od.unit_price * od.quantity * (1 - od.discount)) AS net_revenue FROM orders AS o JOIN order_details AS od ON o.order_id = od.order_id WHERE o.market = 'Europe' AND STRFTIME('%Y', o.order_date) = '2024' GROUP BY month ORDER BY month ASC LIMIT 5000
Rows: 12
Execution ms: 102.82644000835717


In [6]:
# 6) Validation details
for issue in response.validation.issues:
    print(issue.code, "=>", issue.message)


In [7]:
# 7) Explanation output for beginners
print(response.explanation[:2000])


**1. Step‑by‑step SQL logic**

| Step | What the clause does | Why it’s needed for the question |
|------|----------------------|---------------------------------|
| **FROM orders AS o** | Starts with the `orders` table (aliased as `o`). | This is where we locate every order that belongs to Europe. |
| **JOIN order_details AS od ON o.order_id = od.order_id** | Joins each order to its line‑items (`order_details`) using the foreign key `order_id`. | We need the unit price, quantity, and discount for each product sold in an order. |
| **WHERE o.market = 'Europe'** | Filters rows to only those whose market attribute equals “Europe”. | The question asks specifically for Europe’s revenue. |
| **AND STRFTIME('%Y', o.order_date) = '2024'** | Extracts the year from `order_date` and keeps only orders that occurred in 2024. | Limits the data to the calendar year 2024, regardless of month. |
| **SELECT STRFTIME('%Y-%m', o.order_date) AS month** | Formats each order’s date as “YYYY‑MM” (e.g., `


In [8]:
# 8) Visualize query result candidates
import pandas as pd
from ai_sql_assistant.visualization.recommender import render_chart

result_df = pd.DataFrame(response.execution.rows)
result_df.head()


,month,net_revenue
0,2024-01,4.593088e+06
1,2024-02,4.651611e+06
2,2024-03,4.410722e+06
3,2024-04,4.434653e+06
4,2024-05,5.001177e+06


In [9]:
# Try first non-table chart recommendation
if response.visualization_options:
    spec = next((s for s in response.visualization_options if s.chart_type != "table"), response.visualization_options[0])
    fig = render_chart(response.execution.rows, spec)
    fig


## Prompt Engineering Personas
This project includes persona templates:
- Business Analyst
- Finance
- Sales
- HR
- Inventory
- Marketing

Each template shifts SQL priorities while preserving safety rules.


In [10]:
# 9) Compare LangChain SQL generation vs direct Ollama prompting
question = "Top 10 customers by revenue in Germany."
req = QueryRequest(question=question, persona="Sales", user_id="notebook", conversation_id="nb-compare")

res_langchain = assistant.ask(req, approach="langchain", model=settings.models.generator_model)
res_direct = assistant.ask(req, approach="direct", model=settings.models.generator_model)

print("LangChain SQL:\n", res_langchain.execution.sql)
print("\nDirect SQL:\n", res_direct.execution.sql)


LangChain SQL:
 SELECT customer_id, SUM(freight) AS total_revenue FROM orders WHERE ship_country = 'Germany' GROUP BY customer_id ORDER BY total_revenue DESC LIMIT 10

Direct SQL:
 SELECT customer_id, SUM(freight) AS total_revenue FROM orders WHERE ship_country = 'Germany' GROUP BY customer_id ORDER BY total_revenue DESC LIMIT 10


## Benchmark and Evaluation
Benchmark set contains 100 business questions with ground-truth SQL.

Metrics tracked:
- Exact SQL match (normalized)
- Execution accuracy
- Result correctness
- Generation latency
- Execution latency
- Complexity score
- LLM judge rubric (correctness, safety, readability, efficiency)


In [11]:
# 10) Run benchmark (heavy). Uncomment to execute full matrix.
# from pathlib import Path
# from ai_sql_assistant.benchmarking.benchmark import BenchmarkRunner
# runner = BenchmarkRunner(assistant, settings)
# cases = runner.load_cases(Path("benchmarks/benchmark_cases.json"))
# run = runner.run(cases)
# judge = runner.evaluate_with_judge({c.case_id: c for c in cases}, run)
# out_path = runner.save_run(run, judge)
# print(out_path)
# runner.close()


## Failure Analysis
Failure suite covers:
- Hallucinated tables
- Hallucinated columns
- Missing joins
- Ambiguous requests
- Unsafe SQL / prompt injection attempts

Use script:
```bash
uv run python scripts/run_failure_analysis.py
```


## Next Steps
- Launch Streamlit app:
```bash
uv run streamlit run streamlit_app/app.py
```
- Run tests:
```bash
uv run pytest -v
```
- Run end-to-end pipeline:
```bash
uv run python scripts/run_end_to_end.py
```


In [12]:
# Cleanup resources at end of notebook
assistant.close()
